In [1]:
df = spark.read.parquet(
    "s3://airline-dataset-2020-2025/Gold/ML_DATASET/"
)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1785242928929_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
drop_cols = [
    "FlightKey",
    "FlightDate",
    "ReliabilityFeatureScope"
]

df = df.drop(*drop_cols)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
from pyspark.sql.functions import mean

cols = [
    "OriginAirportReliabilityScore",
    "DestAirportReliabilityScore",
    "RouteReliabilityScore"
]

fill_values = {}

for c in cols:
    fill_values[c] = df.select(mean(c)).first()[0]

df = df.na.fill(fill_values)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
categorical_cols = [
    "MarketingAirlineKey",
    "OperatingAirlineKey",
    "DeparturePeriod",
    "ArrivalPeriod",
    "SeasonIndicator",
    "DistanceCategory"
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
numeric_cols = [
    "DepartureHour",
    "ArrivalHour",
    "PeakHourIndicator",
    "WeekendIndicator",
    "Distance",
    "ScheduledElapsedTimeMinutes",
    "CodeshareFlag",
    "IntraStateRouteFlag",
    "AirlineReliabilityScore",
    "OriginAirportReliabilityScore",
    "DestAirportReliabilityScore",
    "RouteReliabilityScore"
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=numeric_cols + [c + "_idx" for c in categorical_cols],
    outputCol="features",
    handleInvalid="skip"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + [assembler]
)

pipeline_model = pipeline.fit(df)

processed_df = pipeline_model.transform(df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
train_df = train_df.drop("DatasetSplit")
valid_df = valid_df.drop("DatasetSplit")
test_df = test_df.drop("DatasetSplit")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
sample_df = processed_df.sample(
    withReplacement=False,
    fraction=0.1,
    seed=42
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
train_df = sample_df.filter("DatasetSplit = 'Train'")
valid_df = sample_df.filter("DatasetSplit = 'Validation'")
test_df  = sample_df.filter("DatasetSplit = 'Test'")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
sample_df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

3989135

In [11]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    maxBins=64,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
feature_names = numeric_cols + [c + "_idx" for c in categorical_cols]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
importance = rf_model.featureImportances.toArray()

feature_importance = list(zip(feature_names, importance))

feature_importance = sorted(
    feature_importance,
    key=lambda x: x[1],
    reverse=True
)

for feature, score in feature_importance:
    print(f"{feature:<40} {score:.6f}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RouteReliabilityScore                    0.179712
DepartureHour                            0.175234
DeparturePeriod_idx                      0.116260
SeasonIndicator_idx                      0.115219
ArrivalHour                              0.110418
AirlineReliabilityScore                  0.097349
ArrivalPeriod_idx                        0.059244
OriginAirportReliabilityScore            0.031835
OperatingAirlineKey_idx                  0.028053
MarketingAirlineKey_idx                  0.027047
DestAirportReliabilityScore              0.018373
CodeshareFlag                            0.017803
ScheduledElapsedTimeMinutes              0.007281
Distance                                 0.007139
PeakHourIndicator                        0.006057
DistanceCategory_idx                     0.001157
WeekendIndicator                         0.001060
IntraStateRouteFlag                      0.000758

In [14]:
selected_features = [
    feature
    for feature, score in feature_importance
    if score > 0.01
]

print(selected_features)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['RouteReliabilityScore', 'DepartureHour', 'DeparturePeriod_idx', 'SeasonIndicator_idx', 'ArrivalHour', 'AirlineReliabilityScore', 'ArrivalPeriod_idx', 'OriginAirportReliabilityScore', 'OperatingAirlineKey_idx', 'MarketingAirlineKey_idx', 'DestAirportReliabilityScore', 'CodeshareFlag']

In [15]:
selected_features = [
    "RouteReliabilityScore",
    "DepartureHour",
    "DeparturePeriod_idx",
    "SeasonIndicator_idx",
    "ArrivalHour",
    "AirlineReliabilityScore",
    "ArrivalPeriod_idx",
    "OriginAirportReliabilityScore",
    "OperatingAirlineKey_idx",
    "MarketingAirlineKey_idx",
    "DestAirportReliabilityScore",
    "CodeshareFlag"
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
from pyspark.ml.feature import VectorAssembler

final_assembler = VectorAssembler(
    inputCols=selected_features,
    outputCol="selected_features"
)

final_df = final_assembler.transform(processed_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
train_df = final_df.filter(processed_df.DatasetSplit == "Train")

valid_df = final_df.filter(processed_df.DatasetSplit == "Validation")

test_df = final_df.filter(processed_df.DatasetSplit == "Test")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
train_df = train_df.drop("DatasetSplit")
valid_df = valid_df.drop("DatasetSplit")
test_df = test_df.drop("DatasetSplit")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="selected_features",
    numTrees=100,
    maxDepth=10,
    maxBins=64,
    seed=42
)

rf_model = rf.fit(train_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
predictions = rf_model.transform(test_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
predictions.select(
    "ArrDel15",
    "prediction",
    "probability"
).show(10, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+----------+----------------------------------------+
|ArrDel15|prediction|probability                             |
+--------+----------+----------------------------------------+
|0       |0.0       |[0.8832284197615219,0.11677158023847803]|
|0       |0.0       |[0.7729012644423764,0.22709873555762364]|
|0       |0.0       |[0.7070551093077345,0.29294489069226554]|
|0       |0.0       |[0.719357364428491,0.28064263557150904] |
|0       |0.0       |[0.8832284197615219,0.11677158023847803]|
|0       |0.0       |[0.7061222559073165,0.2938777440926835] |
|0       |0.0       |[0.719357364428491,0.28064263557150904] |
|0       |0.0       |[0.8832284197615219,0.11677158023847803]|
|0       |0.0       |[0.7061222559073165,0.2938777440926835] |
|0       |0.0       |[0.719357364428491,0.28064263557150904] |
+--------+----------+----------------------------------------+
only showing top 10 rows

In [22]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy:", accuracy.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Accuracy: 0.7781495084893113

In [23]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

print("ROC AUC:", auc.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

ROC AUC: 0.6418654609240868

In [24]:
f1 = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="f1"
)

print("F1 Score:", f1.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

F1 Score: 0.6827662376402378

In [25]:
precision = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

print("Precision:", precision.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Precision: 0.715454150122655

In [26]:
recall = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="weightedRecall"
)

print("Recall:", recall.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Recall: 0.7781495084893113